In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('customer_shopping_behavior.csv')
df.shape

In [ ]:
df.info()
df.describe(include='all').T
# Null check
df.isnull().sum().sort_values(ascending=False)

In [ ]:
missing_before = df['Review Rating'].isnull().sum()

df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(
    lambda x: x.fillna(x.median())
)

print(f"Missing Review Rating values filled: {missing_before}")
print(f"Remaining nulls: {df['Review Rating'].isnull().sum()}")

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('[^0-9a-z_]', '', regex=True)
)
df.columns.tolist()

In [ ]:
# age_group: bins matching the SQL segmentation used later
bins = [17, 25, 40, 60, 100]
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels)
df['age_group'].value_counts()

In [ ]:
# purchase_frequency_days: convert the text frequency field into an approximate
# numeric cadence (days between purchases), useful for later cohort/RFM-style analysis
frequency_map = {
    'Weekly': 7,
    'Fortnightly': 14,
    'Bi-Weekly': 14,
    'Monthly': 30,
    'Every 3 Months': 90,
    'Quarterly': 90,
    'Annually': 365,
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_map)
df[['frequency_of_purchases', 'purchase_frequency_days']].drop_duplicates()

In [ ]:
overlap = pd.crosstab(df['discount_applied'], df['promo_code_used'])
overlap

agreement_rate = (df['discount_applied'] == df['promo_code_used']).mean()
print(f"Agreement rate between the two columns: {agreement_rate:.1%}")

if agreement_rate > 0.95:
    df = df.drop(columns=['promo_code_used'])
    print("Dropped promo_code_used — redundant with discount_applied")

In [ ]:
from sqlalchemy import create_engine

# Update credentials for your local MySQL instance
USER = 'root'
PASSWORD = 'your_password'
HOST = 'localhost'
PORT = 3306
DATABASE = 'customer_behavior'

engine = create_engine(f'mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}')

df.to_sql('customer', con=engine, if_exists='replace', index=False)
print(f"Loaded {len(df)} rows into `customer_behavior.customer`")

In [ ]:
df.groupby('gender')['purchase_amount'].sum().sort_values(ascending=False)

df.groupby('category')['review_rating'].mean().sort_values(ascending=False)